`CPP.run_composit` builds a `df_feat` of **composition** features (iFeature-style descriptors: AAC, DPC, k-mer) instead of positional Part-Split-Scale features. It is the composition counterpart to `CPP.run`, scored with the same discriminative statistics.

- **`composition="aac"`** (amino-acid composition) *is* positional CPP: one-hot amino-acid scales with the whole-part `Segment(1,1)` split, so it yields real `<PART>-Segment(1,1)-<AA>` features **with positions** that the feature map can draw.
- **`composition="dpc"` / `"kmer"`** are **non-positional** (a k-mer is a residue *pair/tuple*, not a per-residue scale): the returned `df_feat` has the k-mer as its `feature` and no `positions`.

In [1]:
import numpy as np
import pandas as pd
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=50)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
cpp = aa.CPP(df_parts=df_parts)
aa.display_df(df_seq, n_rows=10, show_shape=True)

DataFrame shape: (100, 8)


,entry,sequence,label,tmd_start,tmd_stop,jmd_n,tmd,jmd_c
1,Q14802,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,0,37,59,NSPFYYDWHS,LQVGGLICAGVLCAMGIIIVMSA,KCKCKFGQKS
2,Q86UE4,MAARSWQDELAQQAE...SPKQIKKKKKARRET,0,50,72,LGLEPKRYPG,WVILVGTGALGLLLLFLLGYGWA,AACAGARKKR
3,Q969W9,MHRLMGVNSTAAAAA...AIWSKEKDKQKGHPL,0,41,63,FQSMEITELE,FVQIIIIVVVMMVMVVVITCLLS,HYKLSARSFI
4,P53801,MAPGVARGPTPYWRL...GLFKEENPYARFENN,0,97,119,RWGVCWVNFE,ALIITMSVVGGTLLLGIAICCCC,CCRRKRSRKP
5,Q8IUW5,MAPRALPGSAVLAAA...EVPATPVKRERSGTE,0,59,81,NDTGNGHPEY,IAYALVPVFFIMGLFGVLICHLL,KKKGYRCTTE
6,P01135,MVPSAGQLALFALGI...LLKGRTACCHSETVV,0,99,121,AVVAASQKKQ,AITALVVVSIVALAVLIITCVLI,HCCQVRKHCE
7,O43914,MGGLEPCSRLLLLPL...SDVYSDLNTQRPYYK,0,42,64,DCSCSTVSPG,VLAGIVMGDLVLTVLIALAVYFL,GRLVPRGRGA
8,P05556,MNLQPIFWIGLISSV...KSAVTTVVNPKYEGK,0,729,751,ENPECPTGPD,IIPIVAGVVAGIVLIGLALLLIW,KLLMIIHDRR
9,P16234,MGTSHPAFLVLGCLL...DIGIDSSDLVEDSFL,0,527,549,VAPTLRSELT,VAAAVLVLLVIVIISLIVLVVIW,KQKPRYEIRW
10,P50895,MEPPDAPAQARGAPR...SGGARGGSGGFGDEC,0,549,571,TVSPQTSQAG,VAVMAVAVSVGLLLLVVAVFYCV,RRKGGPCCRQ


**AAC (positional).** `run_composit(composition="aac")` returns positional amino-acid features (note the `positions` column) — a genuine CPP `df_feat` drawable by `CPPPlot.feature_map`.

In [2]:
df_aac = cpp.run_composit(labels=labels, composition="aac", n_filter=20, n_jobs=1)
print(f"AAC df_feat: {df_aac.shape}  |  has positions: {'positions' in df_aac.columns}")
aa.display_df(df_aac.round(3), n_rows=10, show_shape=True)

AAC df_feat: (14, 13)  |  has positions: True
DataFrame shape: (14, 13)


/Users/stephanbreimann/Programming/1Packages/wt-368-bootstrap/aaanalysis/feature_engineering/_backend/cpp_run.py:164: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Segment(1,1)-R",Positive,Positive,R,Amino acid R indicator,0.242000,0.053000,0.053000,0.063000,0.053000,0.000000,0.001000,"21,22,23,24,25,...,36,37,38,39,40"
2,"TMD_C_JMD_C-Segment(1,1)-K",Positive,Positive,K,Amino acid K indicator,0.186000,0.036000,0.036000,0.058000,0.054000,0.001000,0.013000,"21,22,23,24,25,...,36,37,38,39,40"
3,"TMD_C_JMD_C-Segment(1,1)-C",Polar,Polar,C,Amino acid C indicator,0.178000,0.046000,-0.046000,0.038000,0.075000,0.002000,0.013000,"21,22,23,24,25,...,36,37,38,39,40"
4,"JMD_N_TMD_N-Segment(1,1)-F",Aromatic,Aromatic,F,Amino acid F indicator,0.173000,0.029000,-0.029000,0.041000,0.051000,0.003000,0.013000,"1,2,3,4,5,6,7,8...,16,17,18,19,20"
5,"TMD-Segment(1,1)-V",Nonpolar,Nonpolar,V,Amino acid V indicator,0.171000,0.051000,0.051000,0.095000,0.084000,0.003000,0.013000,"11,12,13,14,15,...,26,27,28,29,30"
6,"TMD_C_JMD_C-Segment(1,1)-P",Nonpolar,Nonpolar,P,Amino acid P indicator,0.148000,0.021000,-0.021000,0.017000,0.040000,0.011000,0.036000,"21,22,23,24,25,...,36,37,38,39,40"
7,"JMD_N_TMD_N-Segment(1,1)-Q",Polar,Polar,Q,Amino acid Q indicator,0.137000,0.015000,0.015000,0.035000,0.032000,0.018000,0.052000,"1,2,3,4,5,6,7,8...,16,17,18,19,20"
8,"JMD_N_TMD_N-Segment(1,1)-G",Nonpolar,Nonpolar,G,Amino acid G indicator,0.129000,0.030000,0.030000,0.066000,0.057000,0.027000,0.067000,"1,2,3,4,5,6,7,8...,16,17,18,19,20"
9,"JMD_N_TMD_N-Segment(1,1)-Y",Aromatic,Aromatic,Y,Amino acid Y indicator,0.121000,0.015000,-0.015000,0.028000,0.035000,0.038000,0.084000,"1,2,3,4,5,6,7,8...,16,17,18,19,20"
10,"TMD_C_JMD_C-Segment(1,1)-M",Nonpolar,Nonpolar,M,Amino acid M indicator,0.104000,0.014000,0.014000,0.035000,0.026000,0.072000,0.120000,"21,22,23,24,25,...,36,37,38,39,40"


**DPC (non-positional).** `run_composit(composition="dpc")` scores the 400 dipeptides with CPP's statistics and keeps the top `n_filter` by adjusted AUC. The `feature` column holds the dipeptide (e.g. `RR`); there is no `positions` column (a dipeptide has no single position).

In [3]:
df_dpc = cpp.run_composit(labels=labels, composition="dpc", n_filter=15, max_cor=0.5, n_jobs=1)
print(f"DPC df_feat: {df_dpc.shape}  |  has positions: {'positions' in df_dpc.columns}")
aa.display_df(df_dpc.round(3), n_rows=10, show_shape=True)

DPC df_feat: (15, 11)  |  has positions: False
DataFrame shape: (15, 11)


,feature,category,subcategory,scale_name,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh
1,RK,Positive,Positive-Positive,RK,0.154000,0.006000,0.006000,0.011000,0.006000,0.008000,1.000000
2,LV,Nonpolar,Nonpolar-Nonpolar,LV,0.143000,0.013000,0.013000,0.026000,0.021000,0.014000,1.000000
3,LF,Nonpolar,Nonpolar-Aromatic,LF,0.143000,0.010000,-0.010000,0.012000,0.019000,0.014000,1.000000
4,RR,Positive,Positive-Positive,RR,0.123000,0.007000,0.007000,0.017000,0.009000,0.034000,1.000000
5,ML,Nonpolar,Nonpolar-Nonpolar,ML,0.110000,0.006000,0.006000,0.013000,0.004000,0.059000,1.000000
6,VV,Nonpolar,Nonpolar-Nonpolar,VV,0.104000,0.009000,0.009000,0.028000,0.025000,0.073000,1.000000
7,CC,Polar,Polar-Polar,CC,0.102000,0.007000,-0.007000,0.003000,0.020000,0.078000,1.000000
8,VI,Nonpolar,Nonpolar-Nonpolar,VI,0.093000,0.011000,0.011000,0.029000,0.020000,0.108000,1.000000
9,CL,Polar,Polar-Nonpolar,CL,0.087000,0.004000,-0.004000,0.009000,0.012000,0.136000,1.000000
10,IL,Nonpolar,Nonpolar-Nonpolar,IL,0.085000,0.010000,0.010000,0.034000,0.024000,0.143000,1.000000


**k-mer (general).** `composition="kmer"` with `k` (1..4) handles longer motifs. Higher `k` is sparse presence/absence noise, so `min_count` requires a k-mer to occur in at least that many sequences before it can be selected.

In [4]:
df_k3 = cpp.run_composit(labels=labels, composition="kmer", k=3, n_filter=10, min_count=5, n_jobs=1)
print(f"3-mer df_feat: {df_k3.shape}")
aa.display_df(df_k3.round(3), n_rows=10, show_shape=True)

3-mer df_feat: (10, 11)
DataFrame shape: (10, 11)


,feature,category,subcategory,scale_name,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh
1,LVL,Nonpolar,Nonpolar-Nonpolar-Nonpolar,LVL,0.089000,0.005000,0.005000,0.013000,0.007000,0.126000,1.000000
2,KRR,Positive,Positive-Positive-Positive,KRR,0.080000,0.002000,0.002000,0.006000,0.002000,0.168000,1.000000
3,LLV,Nonpolar,Nonpolar-Nonpolar-Nonpolar,LLV,0.076000,0.005000,0.005000,0.014000,0.009000,0.188000,1.000000
4,LLL,Nonpolar,Nonpolar-Nonpolar-Nonpolar,LLL,0.074000,0.007000,-0.007000,0.034000,0.044000,0.201000,1.000000
5,VLV,Nonpolar,Nonpolar-Nonpolar-Nonpolar,VLV,0.070000,0.006000,0.006000,0.017000,0.010000,0.230000,1.000000
6,RKR,Positive,Positive-Positive-Positive,RKR,0.070000,0.002000,0.002000,0.007000,0.002000,0.226000,1.000000
7,VVI,Nonpolar,Nonpolar-Nonpolar-Nonpolar,VVI,0.062000,0.004000,0.004000,0.012000,0.008000,0.285000,1.000000
8,VIV,Nonpolar,Nonpolar-Nonpolar-Nonpolar,VIV,0.061000,0.004000,0.004000,0.012000,0.006000,0.292000,1.000000
9,RRR,Positive,Positive-Positive-Positive,RRR,0.061000,0.003000,0.003000,0.011000,0.003000,0.292000,1.000000
10,LAG,Nonpolar,Nonpolar-Nonpolar-Nonpolar,LAG,0.060000,0.004000,-0.004000,0.000000,0.010000,0.301000,1.000000


**Further parameters.** All composition modes share the CPP-statistic knobs. `label_test` / `label_ref` set the class labels, `parametric` picks the T-test vs Mann-Whitney U p-value, and `start` / `tmd_len` / `jmd_n_len` / `jmd_c_len` set the position labelling of the (positional) amino-acid features.

In [5]:
df_aac2 = cpp.run_composit(labels=labels, composition="aac", n_filter=15, label_test=1, label_ref=0,
                           parametric=False, start=1, tmd_len=20, jmd_n_len=10, jmd_c_len=10, n_jobs=1)
# the same knobs on the general entry point, e.g. a parametric p-value for the k-mer stats:
df_dpc2 = cpp.run_composit(labels=labels, composition="dpc", n_filter=15, label_test=1, label_ref=0,
                           parametric=True, start=1, tmd_len=20, jmd_n_len=10, jmd_c_len=10, n_jobs=1)
print(f"AAC (all params): {df_aac2.shape}  |  run_composit parametric p-value: {df_dpc2.shape}")
aa.display_df(df_aac2.round(3), n_rows=10, show_shape=True)

AAC (all params): (11, 13)  |  run_composit parametric p-value: (15, 11)
DataFrame shape: (11, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Segment(1,1)-R",Positive,Positive,R,Amino acid R indicator,0.242000,0.053000,0.053000,0.063000,0.053000,0.000000,0.000000,"21,22,23,24,25,...,36,37,38,39,40"
2,"TMD_C_JMD_C-Segment(1,1)-K",Positive,Positive,K,Amino acid K indicator,0.186000,0.036000,0.036000,0.058000,0.054000,0.001000,0.010000,"21,22,23,24,25,...,36,37,38,39,40"
3,"TMD_C_JMD_C-Segment(1,1)-C",Polar,Polar,C,Amino acid C indicator,0.178000,0.046000,-0.046000,0.038000,0.075000,0.002000,0.010000,"21,22,23,24,25,...,36,37,38,39,40"
4,"JMD_N_TMD_N-Segment(1,1)-F",Aromatic,Aromatic,F,Amino acid F indicator,0.173000,0.029000,-0.029000,0.041000,0.051000,0.003000,0.010000,"1,2,3,4,5,6,7,8...,16,17,18,19,20"
5,"TMD-Segment(1,1)-V",Nonpolar,Nonpolar,V,Amino acid V indicator,0.171000,0.051000,0.051000,0.095000,0.084000,0.003000,0.010000,"11,12,13,14,15,...,26,27,28,29,30"
6,"TMD_C_JMD_C-Segment(1,1)-P",Nonpolar,Nonpolar,P,Amino acid P indicator,0.148000,0.021000,-0.021000,0.017000,0.040000,0.011000,0.027000,"21,22,23,24,25,...,36,37,38,39,40"
7,"JMD_N_TMD_N-Segment(1,1)-G",Nonpolar,Nonpolar,G,Amino acid G indicator,0.129000,0.030000,0.030000,0.066000,0.057000,0.027000,0.057000,"1,2,3,4,5,6,7,8...,16,17,18,19,20"
8,"TMD-Segment(1,1)-A",Nonpolar,Nonpolar,A,Amino acid A indicator,0.089000,0.020000,0.020000,0.061000,0.061000,0.125000,0.188000,"11,12,13,14,15,...,26,27,28,29,30"
9,"TMD-Segment(1,1)-I",Nonpolar,Nonpolar,I,Amino acid I indicator,0.079000,0.019000,0.019000,0.075000,0.082000,0.172000,0.235000,"11,12,13,14,15,...,26,27,28,29,30"
10,"JMD_N_TMD_N-Segment(1,1)-S",Polar,Polar,S,Amino acid S indicator,0.069000,0.016000,0.016000,0.063000,0.054000,0.233000,0.291000,"1,2,3,4,5,6,7,8...,16,17,18,19,20"
